# Анализ влияния длительности SSL-предобучения на качество fine-tuning (SSL checkpoint transfer)

В данном эксперименте исследуется влияние длительности self-supervised предобучения (SSL) на качество downstream-задачи классификации P300.

Для этого использовались сохранённые состояния энкодера (checkpoint’ы), полученные на различных эпохах SSL-предобучения. Каждый checkpoint рассматривался как инициализация для downstream модели, после чего выполнялся полный fine-tuning (сценарий full_ft) на целевой задаче.

**Дизайн эксперимента:**

SSL checkpoint’ы: эпохи 10, 20, ..., 100
Downstream сценарий: SSL + full fine-tuning
Уровень калибровки: p = 100%
Данные: все субъекты benchmark-выборки (без 4 субъектов, использованных для dev-tuning-small)
Метрики: ROC-AUC, F1

Все остальные компоненты пайплайна (архитектура модели, разбиения, нормализация, гиперпараметры обучения) были зафиксированы, что позволяет изолировать влияние только одного фактора — длительности SSL-предобучения.

**Организация запуска эксперимента**

*В связи с вычислительными ограничениями эксперимент выполнялся по частям. Для этого использовалось разбиение набора SSL-checkpoint’ов на несколько групп.*

*Для запуска каждой части необходимо вручную задать переменную RUN_PART в коде ноутбука.*

**Цель анализа:**
оценить, как изменяется качество downstream-модели в зависимости от эпохи SSL-pretraining и определить, наблюдается ли насыщение (plateau) transfer-качества.

## 1. Импорты

In [2]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import stage5_utils as u

import model_unet as m

## 2. Конфиги

In [3]:
RUNTIME_CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,
}

In [4]:
PATHS = {
    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoints": Path("/kaggle/input/datasets/taisiyaglazova/ssl-checkpoints/checkpoints"),
    "results_root": Path("/kaggle/working/stage5_results"),
}

In [28]:
CHECKPOINTS = {
    10: PATHS['encoder_checkpoints'] / "encoder_epoch_010.pt",
    20: PATHS['encoder_checkpoints'] / "encoder_epoch_020.pt",
    30: PATHS['encoder_checkpoints'] / "encoder_epoch_030.pt",
    40: PATHS['encoder_checkpoints'] / "encoder_epoch_040.pt",
    50: PATHS['encoder_checkpoints'] / "encoder_epoch_050.pt",
    60: PATHS['encoder_checkpoints'] / "encoder_epoch_060.pt",
    70: PATHS['encoder_checkpoints'] / "encoder_epoch_070.pt",
    80: PATHS['encoder_checkpoints'] / "encoder_epoch_080.pt",
    90: PATHS['encoder_checkpoints'] / "encoder_epoch_090.pt",
    100: PATHS['encoder_checkpoints'] / "encoder_epoch_100.pt"
}

## 3. Воспроизводимость

In [8]:
u.set_seed(RUNTIME_CONFIG["seed"])
print("Device:", RUNTIME_CONFIG["device"])

Device: cuda


## 4. Пути

In [9]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": PATHS["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": PATHS["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": PATHS["data_root"] / "bcicompiii-ds2",  
}

In [10]:
GROUPS = ["train", "benchmark", "bcicomp3"]

### Автореестр субъектов

In [11]:
# Автореестр субъектов
def list_subject_ids(group: str):
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        data_dir = DATASETS["bigp3_train"] / "train"
    elif group == "benchmark":
        data_dir = DATASETS["bigp3_benchmark"]/ "benchmark"

    subject_ids = sorted([p.stem for p in data_dir.glob("*.npz")])
    return subject_ids

In [12]:
# Сборка реестра
SUBJECT_REGISTRY = {
    "train": list_subject_ids("train"),
    "benchmark": list_subject_ids("benchmark"),
}

print("Train subjects:", len(SUBJECT_REGISTRY["train"]))
print("Benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("Example train:", SUBJECT_REGISTRY["train"][:5])
print("Example benchmark:", SUBJECT_REGISTRY["benchmark"][:5])


Train subjects: 0
Benchmark subjects: 65
Example train: []
Example benchmark: ['subj_051', 'subj_052', 'subj_053', 'subj_054', 'subj_055']


## Конфиг для FT-сценариев и scratch на Test

In [13]:
DEV_SUBJECTS = [
    'subj_054', 
    'subj_065', 
    'subj_090', 
    'subj_094'
]

In [29]:
ALL_BENCHMARK_SUBJECTS = SUBJECT_REGISTRY["benchmark"]

FILTERED_BENCHMARK_SUBJECTS = [
    s for s in ALL_BENCHMARK_SUBJECTS
    if s not in DEV_SUBJECTS
]

print("Original benchmark subjects:", len(ALL_BENCHMARK_SUBJECTS))
print("After exclusion:", len(FILTERED_BENCHMARK_SUBJECTS))
print("Excluded:", DEV_SUBJECTS)

Original benchmark subjects: 65
After exclusion: 61
Excluded: ['subj_054', 'subj_065', 'subj_090', 'subj_094']


In [35]:
TRANSFER_CONFIG = {
    "tag": "ssl_checkpoint_transfer_full_ft_benchmark",
    "subjects": FILTERED_BENCHMARK_SUBJECTS,
    "group": "benchmark",

    "p_list": [100],

    # фиксируем один FT-сценарий
    "scenario": "ssl_ft",
    "ft_strategy": "full_ft",

    # параметры full_ft
    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
    "warmup_epochs": 3,
}

In [34]:
CHECKPOINT_SPLITS = {
    "part1": [10, 20],
    "part2": [30, 40],
    "part3": [50, 60],
    "part4": [70, 80],
    "part5": [90, 100],
}

In [36]:
RUN_PART = "part5"
SELECTED_EPOCHS = CHECKPOINT_SPLITS[RUN_PART]

SELECTED_CHECKPOINTS = {
    epoch: CHECKPOINTS[epoch]
    for epoch in SELECTED_EPOCHS
}

print("RUN_PART:", RUN_PART)
print("Selected epochs:", SELECTED_EPOCHS)

RUN_PART: part1
Selected epochs: [10, 20]


## Пути для результатов

In [37]:
RUN_DIR = PATHS["results_root"] / TRANSFER_CONFIG["tag"] / RUN_PART

ARTIFACT_DIRS = {
    "root": RUN_DIR,
    "runs": RUN_DIR / "runs",
    "tables": RUN_DIR / "tables",
    "figures": RUN_DIR / "figures",
}

for path in ARTIFACT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

RUN_DIR: /kaggle/working/stage5_results/ssl_checkpoint_transfer_full_ft_benchmark/part1


## Полный перебор checkpoints на Test для full_ft

In [ ]:
all_transfer_dfs = []

for ssl_epoch, ckpt_path in SELECTED_CHECKPOINTS.items():
    print(f"\n{'=' * 80}")
    print(f"SSL checkpoint transfer: epoch {ssl_epoch}")
    print(f"Checkpoint: {ckpt_path}")
    print(f"{'=' * 80}")

    df_epoch = u.run_many(
        subject_list=TRANSFER_CONFIG["subjects"],
        p_list=TRANSFER_CONFIG["p_list"],
        scenario_list=[TRANSFER_CONFIG["scenario"]],
        group=TRANSFER_CONFIG["group"],
        runtime_config=RUNTIME_CONFIG,
        results_root=ARTIFACT_DIRS["runs"] / f"ssl_epoch_{ssl_epoch:03d}",
        encoder_checkpoint=str(ckpt_path),
        ft_strategy_list=[TRANSFER_CONFIG["ft_strategy"]],
        lr_encoder=TRANSFER_CONFIG["lr_encoder"],
        lr_head=TRANSFER_CONFIG["lr_head"],
        weight_decay=TRANSFER_CONFIG["weight_decay"],
        warmup_epochs=TRANSFER_CONFIG["warmup_epochs"],
        save_history=False,
        save_predictions=False,
        save_summary_every=1,
        continue_on_error=True,
    )

    df_epoch["ssl_epoch"] = ssl_epoch
    df_epoch["ssl_checkpoint"] = str(ckpt_path)

    all_transfer_dfs.append(df_epoch)

In [ ]:
import pandas as pd

transfer_df = pd.concat(all_transfer_dfs, ignore_index=True)

summary_path = ARTIFACT_DIRS["tables"] / f"summary_{RUN_PART}.csv"
transfer_df.to_csv(summary_path, index=False)

print("Saved to:", summary_path)
print("Shape:", transfer_df.shape)
display(transfer_df.head())